# 01 — B1: Data Engineer & Feature Architect (CMU CERT r4.2)

**Tujuan:** mengubah log mentah CERT r4.2 menjadi `features_cert.csv.gz` (1 baris = 1 `user_hash` pada 1 `time_key`) yang bersih, ter-pseudonimisasi, dan sesuai **Kontrak A**.


**Struktur folder yang diasumsikan** (notebook ada di `notebooks/`, jadi data dibaca dengan `../`):
```
trailguard/
├── data/raw/          <- hasil ekstrak r4.2 + answers
├── data/processed/    <- output akhir
├── features/          <- builder_cert.py (diisi di akhir)
└── notebooks/         <- notebook ini
```

## Step 1 — Akuisisi & Pemahaman Skema CMU CERT
Tujuan: memastikan file terbaca, memahami kolom tiap file, dan melihat daftar insider serta rentang tanggal serangannya.

In [13]:
import hmac, hashlib, json, re
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

ROOT = Path("..")                       # notebook berada di folder notebooks/
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

TS_FORMAT = "%m/%d/%Y %H:%M:%S"         # format tanggal di log CERT; cek di Step 1b, ubah jika berbeda
KEYS = ["user_hash", "time_key"]


def find_file(name, prefer="r4.2"):
    """Cari file di dalam data/raw (termasuk subfolder). Utamakan path yang memuat 'r4.2'."""
    hits = sorted(RAW.rglob(name))
    preferred = [h for h in hits if prefer and prefer in str(h)]
    hits = preferred or hits
    if not hits:
        raise FileNotFoundError(f"{name} tidak ditemukan di {RAW.resolve()}. Cek hasil ekstrak.")
    return hits[0]


def id_pattern(u) -> str:
    """Tampilkan POLA ID saja (huruf -> A, angka -> 9), mis. 'ABC1234' -> 'AAA9999'.
    Dipakai agar format ID bisa dicek tanpa menampilkan ID asli di output notebook."""
    return re.sub(r"[A-Za-z]", "A", re.sub(r"\d", "9", str(u)))


def mask_emails(s: pd.Series) -> pd.Series:
    """Samarkan bagian depan alamat email, sisakan domainnya: 'nama@dtaa.com' -> '***@dtaa.com'."""
    return s.astype(str).str.replace(r"[^;@\s]+@", "***@", regex=True)


print("Folder data:", RAW.resolve())
print("Contoh isi:", [p.name for p in sorted(RAW.rglob("*.csv"))[:10]])

Folder data: C:\Users\ASUS\OneDrive\Documents\Trailguardproject\data\raw
Contoh isi: ['insiders.csv', 'r2.csv', 'r3.1-1.csv', 'r3.1-2.csv', 'r3.2-1.csv', 'r3.2-2.csv', 'r4.1-1.csv', 'r4.1-2.csv', 'r4.1-3.csv', 'r4.2-1-AAM0658.csv']


In [ ]:
# ===== Pembuat logon.csv SINTETIS (untuk latihan Step 2) =====
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..")                                   # notebook ada di folder notebooks/
OUT_RAW = ROOT / "data" / "raw_toy"                 # folder data tiruan (terpisah dari data asli)
(OUT_RAW / "r4.2").mkdir(parents=True, exist_ok=True)
(OUT_RAW / "answers").mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)

# 1) insiders.csv: pakai yang ASLI bila ada di data/raw (disalin ke raw_toy), kalau tidak buat yang palsu
asli = sorted((ROOT / "data" / "raw").rglob("insiders.csv"))
if asli:
    shutil.copy(asli[0], OUT_RAW / "answers" / "insiders.csv")
    print("insiders.csv ASLI disalin dari:", asli[0].name)
else:
    pd.DataFrame({
        "dataset": [4.2] * 3, "scenario": [1, 2, 3], "details": ["toy"] * 3,
        "user": ["AAA0001", "AAA0002", "AAA0003"],
        "start": ["02/01/2010 08:00:00", "03/15/2010 08:00:00", "04/20/2010 08:00:00"],
        "end":   ["02/15/2010 08:00:00", "03/30/2010 08:00:00", "05/05/2010 08:00:00"],
    }).to_csv(OUT_RAW / "answers" / "insiders.csv", index=False)
    print("insiders.csv ASLI tidak ditemukan, dibuat versi PALSU.")

ins = pd.read_csv(OUT_RAW / "answers" / "insiders.csv")
ins = ins[ins["dataset"].astype(str) == "4.2"].copy()
ins["start"] = pd.to_datetime(ins["start"]).dt.normalize()
ins["end"] = pd.to_datetime(ins["end"]).dt.normalize()
window = {u: (s, e) for u, s, e in zip(ins["user"], ins["start"], ins["end"])}

# 2) daftar user = semua insider + 30 user normal buatan
users = list(window) + [f"NRM{i:04d}" for i in range(30)]
days = pd.date_range("2010-01-04", periods=500, freq="D")      # rentang mirip r4.2

# 3) bangun baris Logon/Logoff
rows = []
for u in users:
    pcs = [f"PC-{rng.integers(1000, 9999)}" for _ in range(2)]
    for d in days:
        if d.dayofweek >= 5 and rng.random() < 0.9:             # akhir pekan jarang aktif
            continue
        menyerang = u in window and window[u][0] <= d <= window[u][1]
        for _ in range(rng.integers(1, 3)):
            hour = int(rng.choice([22, 23, 2, 3])) if menyerang and rng.random() < 0.6 else int(rng.integers(7, 18))
            t_in = d + pd.Timedelta(hours=hour, minutes=int(rng.integers(0, 60)), seconds=int(rng.integers(0, 60)))
            t_out = t_in + pd.Timedelta(hours=int(rng.integers(1, 6)))
            pc = pcs[int(rng.integers(0, 2))]
            rows.append((t_in, u, pc, "Logon"))
            rows.append((t_out, u, pc, "Logoff"))

logon = pd.DataFrame(rows, columns=["date", "user", "pc", "activity"]).sort_values("date").reset_index(drop=True)
logon["date"] = logon["date"].dt.strftime("%m/%d/%Y %H:%M:%S")      # format sama dengan log CERT
logon.insert(0, "id", [f"{{L{i:07d}}}" for i in range(len(logon))])
logon.to_csv(OUT_RAW / "r4.2" / "logon.csv", index=False)

print(f"logon.csv sintetis dibuat: {len(logon):,} baris, {len(users)} user ({len(window)} insider)")
print("Lokasi:", (OUT_RAW / "r4.2" / "logon.csv").resolve())
print("kolom:", list(logon.columns))

### 1a. Intip tiap file (hanya 3 baris, aman untuk file besar)
Kolom `user` **tidak ditampilkan** dan alamat email disamarkan, supaya ID asli tidak tersimpan di output notebook (yang nanti ikut diserahkan). Yang ditampilkan hanya *pola* ID untuk memeriksa formatnya.

In [22]:
for name in ["logon", "device", "file", "email", "http"]:
    path = find_file(f"{name}.csv")
    sample = pd.read_csv(path, nrows=3)
    print(f"\n=== {name}.csv | {path.stat().st_size / 1e6:,.0f} MB ===")
    print("kolom:", list(sample.columns))
    print("pola ID user:", id_pattern(sample["user"].iloc[0]))
    for c in ["to", "cc", "bcc", "from"]:
        if c in sample.columns:
            sample[c] = mask_emails(sample[c])
    display(sample.drop(columns=["user", "content"], errors="ignore"))


=== logon.csv | 59 MB ===
kolom: ['id', 'date', 'user', 'pc', 'activity']
pola ID user: AAA9999


,id,date,pc,activity
0,{X1D9-S0ES98JV-5357PWMI},01/02/2010 06:49:00,PC-6056,Logon
1,{G2B3-L6EJ61GT-2222RKSO},01/02/2010 06:50:00,PC-4275,Logon
2,{U6Q3-U0WE70UA-3770UREL},01/02/2010 06:53:04,PC-4124,Logon



=== device.csv | 29 MB ===
kolom: ['id', 'date', 'user', 'pc', 'activity']
pola ID user: AAA9999


,id,date,pc,activity
0,{J1S3-L9UU75BQ-7790ATPL},01/02/2010 07:21:06,PC-6699,Connect
1,{N7B5-Y7BB27SI-2946PUJK},01/02/2010 07:37:41,PC-6699,Disconnect
2,{U1V9-Z7XT67KV-5649MYHI},01/02/2010 07:59:11,PC-2417,Connect



=== file.csv | 193 MB ===
kolom: ['id', 'date', 'user', 'pc', 'filename', 'content']
pola ID user: AAA9999


,id,date,pc,filename
0,{L9G8-J9QE34VM-2834VDPB},01/02/2010 07:23:14,PC-6699,EYPC9Y08.doc
1,{H0W6-L4FG38XG-9897XTEN},01/02/2010 07:26:19,PC-6699,N3LTSU3O.pdf
2,{M3Z0-O2KK89OX-5716MBIM},01/02/2010 08:12:03,PC-2417,D3D3WC9W.doc



=== email.csv | 1,362 MB ===
kolom: ['id', 'date', 'user', 'pc', 'to', 'cc', 'bcc', 'from', 'size', 'attachments', 'content']
pola ID user: AAA9999


,id,date,pc,to,cc,bcc,from,size,attachments
0,{R3I7-S4TX96FG-8219JWFF},01/02/2010 07:11:45,PC-5758,***@dtaa.com;***@lockheedmartin.com,***@dtaa.com,NaN,***@dtaa.com,25830,0
1,{R0R9-E4GL59IK-2907OSWJ},01/02/2010 07:12:16,PC-6699,***@bellsouth.net,NaN,NaN,***@optonline.net,29942,0
2,{G2B2-A8XY58CP-2847ZJZL},01/02/2010 07:13:00,PC-5758,***@netzero.com,NaN,NaN,***@earthlink.net,28780,0



=== http.csv | 14,536 MB ===
kolom: ['id', 'date', 'user', 'pc', 'url', 'content']
pola ID user: AAA9999


,id,date,pc,url
0,{V1Y4-S2IR20QU-6154HFXJ},01/02/2010 06:55:16,PC-4275,http://msn.com/The_Human_Centipede_First_Seque...
1,{Q5R1-T3EF87UE-2395RWZS},01/02/2010 07:00:13,PC-6056,http://urbanspoon.com/Plunketts_Creek_Loyalsoc...
2,{X9O1-O0XW52VO-5806RPHG},01/02/2010 07:03:46,PC-6056,http://aa.com/Rhodocene/rhodocenium/fhaavatqrf...


### 1b. Rentang waktu log
Kalau muncul error format tanggal di sini, lihat contoh `date` pada 1a lalu sesuaikan `TS_FORMAT` di cell setup.

In [23]:
logon_dates = pd.to_datetime(
    pd.read_csv(find_file("logon.csv"), usecols=["date"])["date"], format=TS_FORMAT
)
print("Rentang log logon:", logon_dates.min(), "s.d.", logon_dates.max())
print("Jumlah hari unik :", logon_dates.dt.normalize().nunique())

Rentang log logon: 2010-01-02 06:49:00 s.d. 2011-05-17 06:43:35
Jumlah hari unik : 501


### 1c. Daftar insider r4.2 (`answers/insiders.csv`)

In [24]:
insiders_raw = pd.read_csv(find_file("insiders.csv", prefer=None))
print("kolom:", list(insiders_raw.columns))
print("dataset yang ada:", insiders_raw["dataset"].astype(str).unique())

# hanya baris untuk r4.2 (file answers memuat banyak versi dataset)
insiders = insiders_raw[insiders_raw["dataset"].astype(str) == "4.2"].copy()
insiders["start"] = pd.to_datetime(insiders["start"])
insiders["end"] = pd.to_datetime(insiders["end"])

print("\nJumlah insider r4.2:", insiders["user"].nunique())
print("Pola ID insider    :", id_pattern(insiders["user"].iloc[0]))
print("Rentang serangan   :", insiders["start"].min(), "s.d.", insiders["end"].max())
display(insiders[["scenario", "start", "end"]].head(10))      # kolom user sengaja tidak ditampilkan
display(insiders["scenario"].value_counts())

kolom: ['dataset', 'scenario', 'details', 'user', 'start', 'end']
dataset yang ada: <StringArray>
['2.0', '3.1', '3.2', '4.1', '4.2', '5.1', '5.2', '6.1', '6.2']
Length: 9, dtype: str

Jumlah insider r4.2: 70
Pola ID insider    : AAA9999
Rentang serangan   : 2010-06-10 07:54:10 s.d. 2011-04-29 20:04:27


,scenario,start,end
8,1,2010-10-23 01:34:19,2010-10-29 05:23:28
9,1,2010-09-10 19:12:01,2010-09-18 02:02:51
10,1,2010-07-30 19:56:44,2010-08-10 05:16:41
11,1,2010-07-13 20:15:23,2010-07-13 21:20:44
12,1,2010-09-21 01:16:22,2010-09-30 04:48:19
13,1,2010-10-06 22:25:52,2010-10-14 06:43:29
14,1,2010-08-11 04:00:08,2010-08-12 23:56:19
15,1,2011-02-04 07:08:00,2011-02-04 07:36:05
16,1,2010-07-22 21:48:43,2010-07-29 01:08:41
17,1,2010-10-02 03:46:16,2010-10-08 22:26:26


scenario
1    30
2    30
3    10
Name: count, dtype: int64

## Step 2 — Pseudonimisasi konsisten & pembersihan awal
Aturan: ID asli (`user`) **tidak boleh** bertahan. Semua tabel dan daftar insider memakai fungsi hash yang **sama**.

> **Catatan notebook:** semua cell Step 2 aman dijalankan ulang. Tabel `insiders` dari Step 1 tidak diubah; hasil ter-hash disimpan di variabel baru **`insiders_h`**, yang dipakai di Step 4.

### 2a. Fungsi hash (persis dari panduan B1) + penyamaan format ID

In [16]:
SALT_KEY = b"TRAILGUARD_GROUP_B_SALT_2026"


def hash_user(user_id: str) -> str:
    return hmac.new(SALT_KEY, str(user_id).strip().encode("utf-8"), hashlib.sha256).hexdigest()[:16]


def clean_user_id(u) -> str:
    '''Hapus spasi dan awalan domain bila ada (mis. 'XXXX/ABC1234' -> 'ABC1234').
    Jika ID di datamu tidak punya awalan, fungsi ini tidak mengubah apa pun.'''
    return str(u).strip().split("/")[-1]


# uji cepat: ID yang sama dengan penulisan berbeda harus menghasilkan hash yang sama
assert hash_user(clean_user_id("XXXX/AAA1111")) == hash_user(clean_user_id("  AAA1111 "))
assert len(hash_user("AAA1111")) == 16
print("Fungsi hash OK")

Fungsi hash OK


### 2b. Fungsi pembersih tabel (hash user + kolom waktu)

In [17]:
def add_user_hash(df: pd.DataFrame) -> pd.DataFrame:
    '''Buat kolom user_hash (hash tiap user unik sekali saja), lalu BUANG kolom 'user' asli.'''
    mapping = {u: hash_user(clean_user_id(u)) for u in df["user"].unique()}
    df["user_hash"] = df["user"].map(mapping)
    return df.drop(columns=["user"])


def add_time_cols(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    '''Buat time_key (YYYY-MM-DD), hour (0-23), is_weekend (0/1), lalu buang kolom tanggal mentah.'''
    ts = pd.to_datetime(df[col], format=TS_FORMAT)
    df["time_key"] = ts.dt.strftime("%Y-%m-%d")
    df["hour"] = ts.dt.hour
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
    return df.drop(columns=[col])


print("Fungsi pembersih siap dipakai")

Fungsi pembersih siap dipakai


### 2c. Terapkan hash yang SAMA ke daftar insider (hasil: `insiders_h`)

In [18]:
insiders_h = insiders.assign(
    user_hash=insiders["user"].map(lambda u: hash_user(clean_user_id(u))),
    start_key=insiders["start"].dt.strftime("%Y-%m-%d"),
    end_key=insiders["end"].dt.strftime("%Y-%m-%d"),
).drop(columns=["user"])                       # jangan simpan ID asli

display(insiders_h[["user_hash", "scenario", "start_key", "end_key"]].head())
print("Jumlah insider ter-hash:", len(insiders_h))

,user_hash,scenario,start_key,end_key
8,f10ff358cb362b82,1,2010-10-23,2010-10-29
9,b47aec21cb111300,1,2010-09-10,2010-09-18
10,6c14f32344258f78,1,2010-07-30,2010-08-10
11,497d0cc64f0bdedb,1,2010-07-13,2010-07-13
12,02bb9feaec2ea5e6,1,2010-09-21,2010-09-30


Jumlah insider ter-hash: 70


In [19]:
contoh = pd.DataFrame({
    "date": ["01/04/2010 07:12:42", "01/09/2010 21:30:00"],   # Senin pagi, Sabtu malam
    "user": ["XXXX/AAA1111", "AAA1111"],                       # ID sama, penulisan beda
    "pc": ["PC-1", "PC-2"],
    "activity": ["Logon", "Logon"],
})

hasil = add_time_cols(add_user_hash(contoh.copy()))
display(hasil)

assert hasil["user_hash"].nunique() == 1                       # dua penulisan -> satu hash
assert list(hasil["time_key"]) == ["2010-01-04", "2010-01-09"]
assert list(hasil["hour"]) == [7, 21]
assert list(hasil["is_weekend"]) == [0, 1]                     # Senin = 0, Sabtu = 1
assert "user" not in hasil.columns and "date" not in hasil.columns
print("Uji mini LULUS")

,pc,activity,user_hash,time_key,hour,is_weekend
0,PC-1,Logon,2eb1d48f1488561d,2010-01-04,7,0
1,PC-2,Logon,2eb1d48f1488561d,2010-01-09,21,1


Uji mini LULUS


### 2d. Uji pembersihan pada satu file (logon)

In [25]:
logon_check = pd.read_csv(find_file("logon.csv"), usecols=["date", "user", "pc", "activity"])
logon_check = add_time_cols(add_user_hash(logon_check))

assert "user" not in logon_check.columns and "date" not in logon_check.columns, "Kolom asli belum terbuang"
assert logon_check["hour"].between(0, 23).all() and logon_check["is_weekend"].isin([0, 1]).all()

print("Baris:", f"{len(logon_check):,}", "| kolom:", list(logon_check.columns))
display(logon_check.head())

Baris: 854,859 | kolom: ['pc', 'activity', 'user_hash', 'time_key', 'hour', 'is_weekend']


,pc,activity,user_hash,time_key,hour,is_weekend
0,PC-6056,Logon,f0b0b7f7bc7b1008,2010-01-02,6,1
1,PC-4275,Logon,e05c38a983ba3298,2010-01-02,6,1
2,PC-4124,Logon,e05c38a983ba3298,2010-01-02,6,1
3,PC-7188,Logon,b54137b8bfad9367,2010-01-02,7,1
4,PC-6699,Logon,ae632f6cfa429984,2010-01-02,7,1


### 2e. Cek terpenting: apakah semua insider ditemukan di log?
Kalau ada yang tidak ketemu, label di Step 4 akan salah (umumnya semua 0). Penyebab paling mungkin: format ID di `insiders.csv` berbeda dengan di `logon.csv`.

In [26]:
found = insiders_h["user_hash"].isin(set(logon_check["user_hash"]))
print(f"Insider r4.2 ditemukan di log: {found.sum()} dari {len(insiders_h)}")
assert found.all(), "Ada insider yang hash-nya tidak ketemu di log. Bandingkan pola ID (Step 1a vs 1c)."
print("Step 2 LULUS")

Insider r4.2 ditemukan di log: 70 dari 70
Step 2 LULUS


## Step 3 — Rekayasa fitur harian (user-day aggregation)

### 3a. Fitur aktivitas mentah harian
Satu fungsi per sumber log. Kolom teks berat (`content`, `url`, `filename`) **tidak dibaca** (`usecols`).
Kolom/format yang perlu dicek ulang dengan hasil Step 1a ditandai `# CEK`.

In [28]:
INTERNAL_DOMAIN = "dtaa.com"      # CEK: domain email internal perusahaan pada data r4.2


def daily_logon() -> pd.DataFrame:
    df = pd.read_csv(find_file("logon.csv"), usecols=["date", "user", "pc", "activity"])
    df = add_time_cols(add_user_hash(df))
    df = df[df["activity"] == "Logon"].copy()
    df["off_hours"] = ((df["hour"] < 7) | (df["hour"] >= 19) | (df["is_weekend"] == 1)).astype(int)
    g = df.groupby(KEYS)
    return pd.DataFrame({
        "logon_count": g.size(),
        "off_hours_logon": g["off_hours"].sum(),
        "unique_pcs": g["pc"].nunique(),
    }).reset_index()


def daily_device() -> pd.DataFrame:
    df = pd.read_csv(find_file("device.csv"), usecols=["date", "user", "activity"])
    df = add_time_cols(add_user_hash(df))
    df = df[df["activity"] == "Connect"]
    return df.groupby(KEYS).size().rename("usb_connect_count").reset_index()


def daily_file() -> pd.DataFrame:
    df = pd.read_csv(find_file("file.csv"), usecols=["date", "user"])
    df = add_time_cols(add_user_hash(df))
    return df.groupby(KEYS).size().rename("file_activity_count").reset_index()


def daily_email() -> pd.DataFrame:
    # CEK: pastikan baris di email.csv memang email yang dikirim user (lihat kolom from/activity di Step 1a)
    df = pd.read_csv(find_file("email.csv"), usecols=["date", "user", "to", "cc", "bcc"])
    recipients = df["to"].fillna("") + ";" + df["cc"].fillna("") + ";" + df["bcc"].fillna("")
    # '@' yang BUKAN diikuti domain internal = penerima eksternal
    pattern = r"@(?!" + re.escape(INTERNAL_DOMAIN) + r"(?:;|$))"
    df["is_external"] = recipients.str.contains(pattern, regex=True).astype(int)
    df = add_time_cols(add_user_hash(df[["date", "user", "is_external"]].copy()))
    return df.groupby(KEYS)["is_external"].sum().rename("external_email_count").reset_index()


def daily_http(chunksize: int = 1_000_000) -> pd.DataFrame:
    """http.csv sangat besar: baca per potongan, langsung ringkas, buang sisanya."""
    parts = []
    for ch in pd.read_csv(find_file("http.csv"), usecols=["date", "user"], chunksize=chunksize):
        ch = add_time_cols(add_user_hash(ch))
        parts.append(ch.groupby(KEYS).size())
    return pd.concat(parts).groupby(level=[0, 1]).sum().rename("http_visit_count").reset_index()

In [29]:
import time

t0 = time.time()
tables = {}
for name, fn in [("logon", daily_logon), ("device", daily_device), ("file", daily_file),
                 ("email", daily_email), ("http", daily_http)]:
    tables[name] = fn()
    print(f"{name:7s} -> {len(tables[name]):>9,} baris user-hari | {time.time() - t0:6.1f} dtk")

logon   ->   330,268 baris user-hari |    5.3 dtk
device  ->    55,717 baris user-hari |    7.8 dtk
file    ->    45,907 baris user-hari |   11.9 dtk
email   ->   326,985 baris user-hari |   47.3 dtk
http    ->   329,845 baris user-hari |  252.8 dtk


In [30]:
daily = reduce(lambda a, b: a.merge(b, on=KEYS, how="outer"), tables.values())
count_cols = [c for c in daily.columns if c not in KEYS]
daily[count_cols] = daily[count_cols].fillna(0.0).astype("float64")

print("Tabel harian:", daily.shape, "| user unik:", daily["user_hash"].nunique())
display(daily.head())
display(daily[count_cols].describe().T)

Tabel harian: (330285, 9) | user unik: 1000


,user_hash,time_key,logon_count,off_hours_logon,unique_pcs,usb_connect_count,file_activity_count,external_email_count,http_visit_count
0,0002b170bbab428a,2010-01-04,1.0,0.0,1.0,0.0,0.0,0.0,10.0
1,0002b170bbab428a,2010-01-05,1.0,0.0,1.0,1.0,0.0,1.0,10.0
2,0002b170bbab428a,2010-01-06,1.0,0.0,1.0,2.0,5.0,1.0,10.0
3,0002b170bbab428a,2010-01-07,1.0,0.0,1.0,2.0,4.0,1.0,10.0
4,0002b170bbab428a,2010-01-08,1.0,0.0,1.0,3.0,3.0,1.0,10.0


,count,mean,std,min,25%,50%,75%,max
logon_count,330285.0,1.424803,0.795422,0.0,1.0,1.0,2.0,9.0
off_hours_logon,330285.0,0.129546,0.558698,0.0,0.0,0.0,0.0,7.0
unique_pcs,330285.0,1.151963,0.642850,0.0,1.0,1.0,1.0,7.0
usb_connect_count,330285.0,0.615647,1.858428,0.0,0.0,0.0,0.0,14.0
file_activity_count,330285.0,1.349080,4.698647,0.0,0.0,0.0,0.0,54.0
external_email_count,330285.0,3.827146,3.547906,0.0,1.0,3.0,6.0,33.0
http_visit_count,330285.0,86.090567,57.263479,0.0,29.0,95.0,114.0,400.0


### 3b. Fitur self-baseline (rolling 7 hari, bebas leakage) + 3c. Cold-start
- `.shift(1)` memastikan nilai hari ini **tidak** masuk ke baseline-nya sendiri.
- Hari pertama seorang user (`NaN`) diisi `0.0`, sesuai aturan wajib.

> **Catatan untuk didiskusikan dengan tim :** baris hanya ada pada hari ketika user punya aktivitas, jadi "7 baris sebelumnya" belum tentu "7 hari kalender". Selain itu, bila riwayat 7 hari konstan (std = 0), z-score bisa sangat besar karena dibagi `1e-5`. Cek di QC (Step 4) dan laporkan ke B0/B3 jika dirasa mengganggu.

In [31]:
BASE_FEATURES = {                      # kolom mentah -> nama pendek untuk fitur turunan
    "logon_count": "logon",
    "off_hours_logon": "off_hours",
    "usb_connect_count": "usb",
    "file_activity_count": "file",
    "external_email_count": "email",
    "http_visit_count": "http",
}


def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["user_hash", "time_key"]).reset_index(drop=True)
    new_cols = []
    for col, short in BASE_FEATURES.items():
        shifted = df.groupby("user_hash")[col].shift(1)              # hanya hari-hari sebelumnya
        by_user = shifted.groupby(df["user_hash"])
        roll_mean = by_user.transform(lambda s: s.rolling(7, min_periods=1).mean())
        roll_std = by_user.transform(lambda s: s.rolling(7, min_periods=2).std()).fillna(0.0)

        df[f"{short}_zscore_7d"] = (df[col] - roll_mean) / (roll_std + 1e-5)
        df[f"{short}_spike_ratio"] = (df[col] + 1.0) / (roll_mean + 1.0)
        new_cols += [f"{short}_zscore_7d", f"{short}_spike_ratio"]

    df[new_cols] = df[new_cols].fillna(0.0)                           # cold-start: NaN -> 0.0
    return df


feat = add_rolling_features(daily)
print(feat.shape)
display(feat.head(10))

(330285, 21)


,user_hash,time_key,logon_count,off_hours_logon,unique_pcs,usb_connect_count,file_activity_count,external_email_count,http_visit_count,logon_zscore_7d,logon_spike_ratio,off_hours_zscore_7d,off_hours_spike_ratio,usb_zscore_7d,usb_spike_ratio,file_zscore_7d,file_spike_ratio,email_zscore_7d,email_spike_ratio,http_zscore_7d,http_spike_ratio
0,0002b170bbab428a,2010-01-04,1.0,0.0,1.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
1,0002b170bbab428a,2010-01-05,1.0,0.0,1.0,1.0,0.0,1.0,10.0,0.0,1.0,0.0,1.0,100000.000000,2.000000,0.000000,1.000000,100000.000000,2.000000,0.0,1.0
2,0002b170bbab428a,2010-01-06,1.0,0.0,1.0,2.0,5.0,1.0,10.0,0.0,1.0,0.0,1.0,2.121290,2.000000,500000.000000,6.000000,0.707097,1.333333,0.0,1.0
3,0002b170bbab428a,2010-01-07,1.0,0.0,1.0,2.0,4.0,1.0,10.0,0.0,1.0,0.0,1.0,0.999990,1.500000,0.808288,1.875000,0.577340,1.200000,0.0,1.0
4,0002b170bbab428a,2010-01-08,1.0,0.0,1.0,3.0,3.0,1.0,10.0,0.0,1.0,0.0,1.0,1.827796,1.777778,0.285175,1.230769,0.499990,1.142857,0.0,1.0
5,0002b170bbab428a,2010-01-11,1.0,0.0,1.0,1.0,5.0,0.0,10.0,0.0,1.0,0.0,1.0,-0.526230,0.769231,1.129363,1.764706,-1.788814,0.555556,0.0,1.0
6,0002b170bbab428a,2010-01-12,1.0,0.0,1.0,2.0,1.0,1.0,10.0,0.0,1.0,0.0,1.0,0.476727,1.200000,-0.791384,0.521739,0.645485,1.200000,0.0,1.0
7,0002b170bbab428a,2010-01-13,1.0,0.0,1.0,2.0,0.0,0.0,10.0,0.0,1.0,0.0,1.0,0.439151,1.166667,-1.155488,0.280000,-1.463820,0.583333,0.0,1.0
8,0002b170bbab428a,2010-01-14,1.0,0.0,1.0,2.0,0.0,0.0,10.0,0.0,1.0,0.0,1.0,0.207017,1.050000,-1.155488,0.280000,-1.463820,0.583333,0.0,1.0
9,0002b170bbab428a,2010-01-15,1.0,0.0,1.0,1.0,0.0,1.0,10.0,0.0,1.0,0.0,1.0,-1.732021,0.666667,-1.155488,0.280000,0.801769,1.272727,0.0,1.0


## Step 4 — Pelabelan & Quality Check
`label = 1` bila `user_hash` adalah insider **dan** `time_key` berada di antara `start_key` s.d. `end_key` (inklusif).
Urutan kolom mengikuti **Kontrak A**: `time_key`, `user_hash`, `label`, lalu fitur `float64`.

In [32]:
def add_labels(df: pd.DataFrame, insiders: pd.DataFrame) -> pd.DataFrame:
    df["label"] = 0
    for _, r in insiders.iterrows():
        m = (df["user_hash"] == r["user_hash"]) & (df["time_key"] >= r["start_key"]) & (df["time_key"] <= r["end_key"])
        df.loc[m, "label"] = 1
    return df


def finalize_columns(df: pd.DataFrame):
    feature_cols = [c for c in df.columns if c not in ("time_key", "user_hash", "label")]
    df = df[["time_key", "user_hash", "label"] + feature_cols].copy()
    df[feature_cols] = df[feature_cols].astype("float64")
    df["label"] = df["label"].astype(int)
    df = df.sort_values(["time_key", "user_hash"]).reset_index(drop=True)
    return df, feature_cols


final_df, feature_cols = finalize_columns(add_labels(feat.copy(), insiders_h))
print(final_df.shape, "| jumlah fitur:", len(feature_cols))
print("Baris label=1:", int(final_df["label"].sum()), f"({final_df['label'].mean():.4%})")
display(final_df.head())

(330285, 22) | jumlah fitur: 19
Baris label=1: 1364 (0.4130%)


,time_key,user_hash,label,logon_count,off_hours_logon,unique_pcs,usb_connect_count,file_activity_count,external_email_count,http_visit_count,logon_zscore_7d,logon_spike_ratio,off_hours_zscore_7d,off_hours_spike_ratio,usb_zscore_7d,usb_spike_ratio,file_zscore_7d,file_spike_ratio,email_zscore_7d,email_spike_ratio,http_zscore_7d,http_spike_ratio
0,2010-01-02,01a731e941976711,0,1.0,1.0,1.0,2.0,13.0,2.0,162.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2010-01-02,020b141eddf22985,0,2.0,2.0,1.0,0.0,0.0,1.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2010-01-02,0b92d31c684a9f5d,0,1.0,1.0,1.0,0.0,0.0,0.0,29.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2010-01-02,0c151dca68bea379,0,2.0,2.0,1.0,2.0,0.0,4.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2010-01-02,1602ce7683ba8a04,0,1.0,1.0,1.0,0.0,0.0,1.0,29.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
# ---- Quality check (Kontrak A & B) ----
assert final_df.isna().sum().sum() == 0, "Masih ada NaN"
assert not np.isinf(final_df[feature_cols].to_numpy()).any(), "Ada nilai inf"
assert list(final_df.columns[:3]) == ["time_key", "user_hash", "label"], "Urutan kolom metadata salah"
assert all(final_df[c].dtype == "float64" for c in feature_cols), "Ada fitur bukan float64"
assert final_df["time_key"].is_monotonic_increasing, "time_key belum terurut"
print("Cek dasar LULUS")

# ---- Proporsi label pada split temporal 70/15/15 (Kontrak B) ----
n = len(final_df)
i1, i2 = int(n * 0.70), int(n * 0.85)
splits = {"train": final_df.iloc[:i1], "val": final_df.iloc[i1:i2], "test": final_df.iloc[i2:]}
for name, part in splits.items():
    print(f"{name:5s} baris={len(part):>8,} | {part['time_key'].min()} s.d. {part['time_key'].max()} "
          f"| label=1: {int(part['label'].sum()):>6,} ({part['label'].mean():.3%})")
    # Wajib: train, val, test sama-sama punya kasus positif. Jika tidak, laporkan ke tim.

# ---- Lihat sebaran z-score (cari nilai ekstrem) ----
display(final_df[[c for c in feature_cols if c.endswith("zscore_7d")]].describe().T[["min", "50%", "max"]])

Cek dasar LULUS
train baris= 231,199 | 2010-01-02 s.d. 2010-12-06 | label=1:    931 (0.403%)
val   baris=  49,543 | 2010-12-06 s.d. 2011-02-24 | label=1:    282 (0.569%)
test  baris=  49,543 | 2011-02-24 s.d. 2011-05-17 | label=1:    151 (0.305%)


,min,50%,max
logon_zscore_7d,-400000.0,0.000000,400000.0
off_hours_zscore_7d,-400000.0,0.000000,400000.0
usb_zscore_7d,-900000.0,0.000000,1000000.0
file_zscore_7d,-2700000.0,0.000000,2700000.0
email_zscore_7d,-1400000.0,-0.038915,1100000.0
http_zscore_7d,-40000000.0,0.000000,1000000.0


In [34]:
final_df.to_csv(PROCESSED / "features_cert.csv.gz", index=False)
with open(PROCESSED / "cert_feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

print("Tersimpan:", (PROCESSED / "features_cert.csv.gz").resolve())
print("Ukuran   :", f"{(PROCESSED / 'features_cert.csv.gz').stat().st_size / 1e6:.1f} MB")

Tersimpan: C:\Users\ASUS\OneDrive\Documents\Trailguardproject\data\processed\features_cert.csv.gz
Ukuran   : 10.9 MB
